<a href="https://colab.research.google.com/github/Breyo20/BreiderPOO/blob/main/Taller_RPGManager_Herencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from abc import ABC, abstractmethod

class Personaje(ABC):

    def __init__(self, nombre, nivel, vida_max):
        self.nombre = nombre
        self.nivel = nivel
        self.puntosVidaMax = vida_max
        self.puntosVida = vida_max

    def recibirDano(self, dano):
        self.puntosVida -= dano
        if self.puntosVida < 0:
            self.puntosVida = 0
        print(f"{self.nombre} recibió {dano} de daño. HP actual: {self.puntosVida}/{self.puntosVidaMax}")

    def estaVivo(self):
        return self.puntosVida > 0

    @abstractmethod
    def atacar(self, objetivo):
        pass

    @abstractmethod
    def getTipoPersonaje(self):
        pass

    def __str__(self):
        return f"[{self.getTipoPersonaje()}] {self.nombre} Nv.{self.nivel} | HP: {self.puntosVida}/{self.puntosVidaMax}"

In [12]:
class Guerrero(Personaje, Habilidoso, Equipable):

    def __init__(self, nombre, nivel, vida_max, fuerza, defensa):
        super().__init__(nombre, nivel, vida_max)
        self.fuerza = fuerza
        self.defensa = defensa

        self.itemEquipado = "Sin equipo"
        self.costoHabilidad = 30

    def atacar(self, objetivo):
        print(f"{self.nombre} ataca a {objetivo.nombre} con fuerza {self.fuerza}")
        objetivo.recibirDano(self.fuerza)

    def getTipoPersonaje(self):
        return "Guerrero"

    def usarEscudo(self):
        print(f"{self.nombre} bloquea con defensa {self.defensa}")

    #  HABILIDOSO
    def usar_habilidad_especial(self, obj):
        print(f"{self.nombre} usa Golpe Devastador sobre {obj.nombre} (50 daño)")
        obj.recibirDano(50)

    def get_costo_habilidad(self):
        return self.costoHabilidad

    def get_nombre_habilidad(self):
        return "Golpe Devastador"

    #  EQUIPABLE
    def equipar_item(self, item):
        self.itemEquipado = item
        print(f"{self.nombre} ha equipado: {item}")

    def get_item_equipado(self):
        return self.itemEquipado

In [13]:
class Mago(Personaje, Habilidoso, Sanador):

    def __init__(self, nombre, nivel):
        vida_max = 80 + nivel * 10
        super().__init__(nombre, nivel, vida_max)
        self.manaMax = 80 + nivel * 10
        self.mana = self.manaMax

    def atacar(self, objetivo):
        if self.mana >= 20:
            dano = 25 + self.nivel * 5
            print(f"{self.nombre} lanza un hechizo a {objetivo.nombre} causando {dano} de daño")
            objetivo.recibirDano(dano)
            self.mana -= 20
        else:
            print(f"{self.nombre} no puede atacar: mana insuficiente")

    def getTipoPersonaje(self):
        return "Mago"

    def recuperarMana(self, cantidad):
        self.mana += cantidad
        if self.mana > self.manaMax:
            self.mana = self.manaMax
        print(f"{self.nombre} recupera mana. Mana actual: {self.mana}/{self.manaMax}")

    #  HABILIDOSO
    def usar_habilidad_especial(self, obj):
        if self.mana >= 20:
            print(f"{self.nombre} lanza Bola de Fuego a {obj.nombre} (40 daño)")
            obj.recibirDano(40)
            self.mana -= 20
        else:
            print(f"{self.nombre} no puede usar Bola de Fuego: mana insuficiente")

    def get_costo_habilidad(self):
        return 20

    def get_nombre_habilidad(self):
        return "Bola de Fuego"

    #  SANADOR
    def sanar(self, obj):
        nueva_vida = obj.puntosVida + 25
        if nueva_vida > obj.puntosVidaMax:
            nueva_vida = obj.puntosVidaMax
        obj.puntosVida = nueva_vida

        print(f"{self.nombre} sana a {obj.nombre} (+25 HP)")

    def get_potencia_sanacion(self):
        return 25

In [14]:
class Arquero(Personaje, Equipable):

    def __init__(self, nombre, nivel):
        vida_max = 70 + nivel * 8
        super().__init__(nombre, nivel, vida_max)
        self.flechas = 10 + nivel * 2
        self.alcance = 30

        self.itemEquipado = "Arco basico"

    def atacar(self, objetivo):
        if self.flechas > 0:
            dano = 12 + self.nivel * 4

            #  bono por equipo
            if self.itemEquipado != "Arco basico":
                dano += 5

            print(f"{self.nombre} dispara una flecha a {objetivo.nombre} causando {dano} de daño")
            objetivo.recibirDano(dano)
            self.flechas -= 1
        else:
            print(f"{self.nombre} no puede atacar: no le quedan flechas")

    def getTipoPersonaje(self):
        return "Arquero"

    def recargarFlechas(self, cantidad):
        self.flechas += cantidad
        print(f"{self.nombre} recarga flechas. Flechas actuales: {self.flechas}")

    #  EQUIPABLE
    def equipar_item(self, item):
        self.itemEquipado = item
        print(f"{self.nombre} ha equipado: {item}")

    def get_item_equipado(self):
        return self.itemEquipado

In [ ]:
# Crear héroes
thorin = Guerrero("Thorin", 3, 100, 20, 10)
gandalf = Mago("Gandalf", 5)
legolas = Arquero("Legolas", 4)

heroes = [thorin, gandalf, legolas]

# Crear enemigo
orco = Guerrero("Orco", 1, 80, 15, 5)

#  FASE 1 — Equipar
thorin.equipar_item("Espada Legendaria")
legolas.equipar_item("Arco Elfico")

turno = 1

#  FASE 2 — Batalla
while orco.estaVivo():

    print(f"\n--- Turno {turno} ---")

    for h in heroes:

        if not orco.estaVivo():
            break

        #  turno 2: usar habilidad
        if turno == 2 and isinstance(h, Habilidoso):
            h.usar_habilidad_especial(orco)

        if orco.estaVivo():
            h.atacar(orco)

    turno += 1

print(f"\n Batalla terminada en {turno-1} turnos\n")

# FASE 3 — Sanación
for h in heroes:

    if isinstance(h, Sanador):

        # buscar el héroe con menos vida
        objetivo = heroes[0]
        for p in heroes:
            if p.puntosVida < objetivo.puntosVida:
                objetivo = p

        h.sanar(objetivo)

# Estado final
print("\nEstado final:")
for h in heroes:
    print(h)
print(orco)

In [18]:
thorin = Guerrero("Thorin", 3, 100, 20, 10)
gandalf = Mago("Gandalf", 5)
legolas = Arquero("Legolas", 4)

print(thorin)
print(gandalf)
print(legolas)

[Guerrero] Thorin Nv.3 | HP: 100/100
[Mago] Gandalf Nv.5 | HP: 130/130
[Arquero] Legolas Nv.4 | HP: 102/102


In [ ]:
# Crear héroes
heroes = [
    Guerrero("Thorin", 3, 100, 20, 10),
    Mago("Gandalf", 5),
    Arquero("Legolas", 4)
]

# Crear enemigo
orco = Guerrero("Orco", 1, 80, 15, 5)

turno = 1

# Bucle de batalla
while orco.estaVivo():
    print(f"\n--- Turno {turno} ---")

    for h in heroes:
        if orco.estaVivo():
            h.atacar(orco)
        else:
            break

    turno += 1

# Resultado final
print(f"\n Batalla terminada en {turno-1} turnos\n")

print("Estado final:")
for h in heroes:
    print(h)
print(orco)

In [ ]:
from abc import ABC, abstractmethod

class Habilidoso(ABC):

    @abstractmethod
    def usar_habilidad_especial(self, obj):
        pass

    @abstractmethod
    def get_costo_habilidad(self):
        pass

    @abstractmethod
    def get_nombre_habilidad(self):
        pass


class Equipable(ABC):

    @abstractmethod
    def equipar_item(self, item):
        pass

    @abstractmethod
    def get_item_equipado(self):
        pass


class Sanador(ABC):

    @abstractmethod
    def sanar(self, objetivo):
        pass

    @abstractmethod
    def get_potencia_sanacion(self):
        pass